In [1]:
from pathlib import Path
import threading
from http.server import HTTPServer, SimpleHTTPRequestHandler
from google.colab import output

# ------------------------------------------------------------
# SOL SCIENCE WEBSITE (HTML + CSS + JavaScript in one string)
# ------------------------------------------------------------

website = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>SunbYte Observatory Labs (SOL)</title>

<style>
    /* ---------- General ---------- */
    html { scroll-behavior: smooth; scroll-padding-top: 70px; }
    body { font-family: Arial, sans-serif; background: #f0f4ff; color: #172554; text-align: center; margin: 0; }
    section { padding: 30px 20px; }
    footer { background: #111827; color: white; padding: 20px; }

    /* ---------- Navigation bar ---------- */
    nav { background: #111827; padding: 18px; position: sticky; top: 0; z-index: 10; }
    nav a { color: white; margin: 15px; text-decoration: none; font-weight: bold; }
    nav a:hover { color: #60a5fa; }

    /* ---------- Header ---------- */
    header { background: linear-gradient(135deg, #A34700, #E68E00); color: white; padding: 60px 20px; }

    /* ---------- Buttons ---------- */
    button { background: #D16200; color: white; border: none; padding: 12px 20px; border-radius: 8px; cursor: pointer; margin: 6px; font-size: 14px; }
    button:hover { background: #3b82f6; }
    button.active { background: #1e40af; }

    /* ---------- Activity cards on the home page ---------- */
    .cards { display: flex; flex-wrap: wrap; gap: 25px; justify-content: center; max-width: 1000px; margin: 30px auto; }
    .card { background: white; border-radius: 16px; padding: 25px; width: 290px; max-width: 100%; text-decoration: none; color: #172554; box-shadow: 0 5px 15px #00000020; }
    .card:hover { transform: translateY(-5px); }
    .card h3 { margin-top: 0; }

    /* ---------- Activity pages ---------- */
    .activity { background: #e0eaff; }
    .panel { max-width: 800px; margin: 20px auto; background: white; padding: 25px; border-radius: 15px; box-shadow: 0 4px 12px #00000015; }
    .info { background: #eff6ff; padding: 15px; border-radius: 10px; line-height: 1.6; margin-top: 15px; text-align: left; }
    .predict { background: #fef3c7; padding: 12px; border-radius: 10px; margin-bottom: 15px; }
    .formula { background: #172554; color: white; padding: 15px; border-radius: 10px; font-size: 22px; max-width: 400px; margin: 20px auto; }
    label { display: block; font-weight: bold; margin: 12px 0 4px 0; }
    input[type="range"] { width: 80%; }
    select { padding: 8px; font-size: 14px; }

    /* ---------- Telescope view of the Sun ---------- */
    .sun-view { height: 280px; max-width: 600px; margin: 20px auto; border-radius: 15px; background: #020617; display: flex; justify-content: center; align-items: center; position: relative; overflow: hidden; }
    .sun { width: 100px; height: 100px; border-radius: 50%; background: radial-gradient(circle, #fff7ae, #fbbf24, #f97316); box-shadow: 0 0 35px #facc15; }
    .cloud { display: none; position: absolute; width: 180px; height: 55px; background: #94a3b8; border-radius: 50px; top: 35%; left: 20%; box-shadow: 40px -20px 0 5px #94a3b8, 90px 0 0 #94a3b8; }

    /* Weather changes: JavaScript adds one of these classes to .sun-view */
    .cloudy .cloud { display: block; }
    .rainy { background: #334155; }
    .rainy .cloud { display: block; }
    .hazy .sun { filter: blur(8px); opacity: 0.6; }
    .turbulent .sun { animation: wobble 0.25s infinite alternate; }
    @keyframes wobble { from { margin-left: -8px; } to { margin-left: 8px; } }

    .three { display: flex; gap: 12px; flex-wrap: wrap; justify-content: center; }
    .three div { flex: 1 1 180px; background: #eff6ff; padding: 12px; border-radius: 10px; }
    .row { display: flex; flex-wrap: wrap; gap: 10px; justify-content: center; }
    .scene { flex: 1 1 200px; height: 280px; position: relative; overflow: hidden; border-radius: 12px; background: linear-gradient(#020617, #1e3a8a 60%, #60a5fa); }
    .ground { position: absolute; bottom: 0; left: 0; right: 0; height: 10%; background: #166534; }
    .balloon { position: absolute; left: calc(50% - 25px); bottom: 4%; width: 50px; transition: bottom 2s; }
    .balloon-top { width: 50px; height: 60px; background: #D16200; border-radius: 50% 50% 45% 45%; }
    .basket { width: 14px; height: 12px; background: #78350f; margin: 8px auto 0 auto; }
    .part { cursor: pointer; }
    .part:hover { opacity: 0.7; }
    .chosen { stroke: #facc15; stroke-width: 4; }
    canvas { max-width: 100%; border-radius: 12px; background: black; }
</style>
</head>

<body>

<!-- ================= NAVIGATION ================= -->
<nav>
    <a href="#home">Home</a>
    <a href="#activities">Activities</a>
    <a href="#spectrometer">Spectrometer</a>
    <a href="#observatory">Observatory</a>
    <a href="#fabry">Fabry-Perot</a>
</nav>

<!-- ================= HEADER ================= -->
<header id="home">
    <h1>Welcome to SunbYte Observatory Labs</h1>
    <p>Hands-on workshops in solar physics and space engineering.</p>
    <button onclick="location.href='#activities'">Explore the Activities</button>
</header>

<!-- ================= HOME: TWO ACTIVITY CARDS ================= -->
<section id="activities">
    <h2>Choose an activity</h2>
    <div class="cards">
        <a href="#spectrometer" class="card">
            <h3>Spectrometer</h3>
            <p>Change the integration time and see how the spectrum improves, then spoils.</p>
        </a>
        <a href="#observatory" class="card">
            <h3>Observatory Placement</h3>
            <p>Where should we build a telescope? Explore altitude, seeing, extinction, light pollution and moonlight.</p>
        </a>
        <a href="#fabry" class="card">
            <h3>Fabry-Perot Interferometer</h3>
            <p>Explore interference rings by changing mirror spacing, refractive index, angle and reflectivity.</p>
        </a>
    </div>

    <div class="panel">
        <h3>About SOL and Project SunbYte</h3>
        <p><strong>SunbYte Observatory Labs (SOL)</strong> is a workshop series run with the Space Universities Network and the UK Space Agency. It gets undergraduate students involved in solar physics and space engineering.</p>
        <p><strong>Project SunbYte</strong> is a student-led project at the University of Sheffield. It lifts a low-cost solar telescope on a high-altitude balloon, above most of the atmosphere.</p>
        <p><a href="https://sites.google.com/sheffield.ac.uk/project-sunbyte" target="_blank">Visit the Project SunbYte website</a></p>

    <div class="panel" style="text-align:left">
        <h3>What you will learn</h3>
        <ul>
            <li>How imagers, spectrometers and Fabry-Perot interferometers are used.</li>
            <li>How to operate a spectrograph and why different light sources give different spectra.</li>
            <li>The principles of solar observation with a ground-based telescope.</li>
            <li>How Fabry-Perot interferometers measure solar features with sub-angstrom accuracy.</li>
        </ul>
    </div>

    <div class="panel">
        <h3>Three instruments</h3>
        <div class="three">
            <div><strong>Imager</strong><br>Takes a picture. Shows where things are.</div>
            <div><strong>Spectrometer</strong><br>Splits light into wavelengths. Shows what the light is made of.</div>
            <div><strong>Fabry-Perot</strong><br>Picks out a very narrow range of wavelengths. Shows fine detail.</div>
        </div>
    </div>
</section>

<!-- ================= ACTIVITY: SPECTROMETER ================= -->
<section id="spectrometer" class="activity">
    <h2>Spectrometer: Integration Time</h2>
    <p>Integration time is how long the detector collects photons for each measurement.</p>

    <div class="panel">
        <div class="predict"><strong>Challenge:</strong> Find the integration time that gives a clear spectrum without clipping.</div>
        <canvas id="specCanvas" width="600" height="300"></canvas>

        <label>Light source</label>
        <select id="srcSelect" onchange="drawSpectrum()">
            <option value="sun">The Sun</option>
            <option value="hydrogen">Hydrogen lamp</option>
            <option value="sodium">Sodium lamp</option>
            <option value="led">White LED</option>
        </select>

        <label>Integration time: <span id="tValue">10</span> ms</label>
        <input type="range" id="tSlider" min="1" max="100" step="1" value="10" oninput="drawSpectrum()">

        <div class="info" id="specInfo"></div>
    </div>

    <div class="panel">
        <h3>Key ideas</h3>
        <p><strong>Short exposure:</strong> few photons, so the signal is noisy.</p>
        <p><strong>Long exposure:</strong> more photons, so the signal is smoother.</p>
        <p><strong>Clipping (saturation):</strong> if a pixel collects too many photons it hits its limit and the real peak height is lost.</p>
        <p><strong>Absorption lines</strong> are dips (the Sun). <strong>Emission lines</strong> are peaks (lamps).</p>
        <p><small>Simplified teaching values, not real detector data.</small></p>
    </div>

    <button onclick="location.href='#activities'">Back to activities</button>
</section>

<!-- ================= ACTIVITY: OBSERVATORY ================= -->
<section id="observatory" class="activity">
    <h2>Observatory Placement</h2>
    <p>Good observatories are high, dry, dark and have steady air.</p>

    <!-- Part A: balloon -->
    <div class="panel">
        <h3>Part A: Rise above the atmosphere</h3>
        <div class="predict"><strong>Predict first:</strong> How will the Sun look from the ground, a little higher up, and from balloon height?</div>
        <div class="row">
            <div class="scene">
                <div class="ground"></div>
                <div class="balloon" id="balloon"><div class="balloon-top"></div><div class="basket"></div></div>
            </div>
            <div class="sun-view" style="flex:1 1 200px; margin:0; height:280px">
                <div class="sun" id="sun2"></div>
            </div>
        </div>
        <div>
            <button class="lvlBtn active" id="lvlGround" onclick="setLevel('ground', this)">Ground</button>
            <button class="lvlBtn" onclick="setLevel('low', this)">A little above ground</button>
            <button class="lvlBtn" onclick="setLevel('high', this)">Balloon height</button>
        </div>
        <div class="info" id="levelInfo"></div>
        <p><small>SunbYte uses a high-altitude balloon. This picture is simplified.</small></p>
    </div>

    <!-- Part A: the Sun and the weather -->
    <div class="panel">
        <h3>Part B: Weather and the Sun</h3>
        <div class="predict"><strong>Predict first:</strong> Which weather will make the Sun look worst?</div>

        <div class="sun-view" id="sunView">
            <div class="cloud"></div>
            <div class="sun" id="sun"></div>
        </div>

        <label>Magnification: <span id="zoomValue">1</span>x</label>
        <input type="range" id="zoom" min="1" max="3" step="0.1" value="1" oninput="changeZoom()">

        <div>
            <button class="weatherBtn active" onclick="setWeather('clear', this)">Clear</button>
            <button class="weatherBtn" onclick="setWeather('cloudy', this)">Cloudy</button>
            <button class="weatherBtn" onclick="setWeather('rainy', this)">Rainy</button>
            <button class="weatherBtn" onclick="setWeather('hazy', this)">Hazy</button>
            <button class="weatherBtn" onclick="setWeather('turbulent', this)">Turbulent air</button>
        </div>

        <div class="info" id="weatherInfo"></div>
        <p><small>Simplified simulation. Never look at the Sun without certified solar equipment.</small></p>
    </div>

    <!-- Part B: the night sky -->
    <div class="panel">
        <h3>Part C: Night sky challenge</h3>
        <div class="predict"><strong>Challenge:</strong> Find the settings that show the MOST stars. Where would you build an observatory?</div>

        <canvas id="skyCanvas" width="600" height="300"></canvas>

        <label>Light pollution: <span id="lpValue">0</span> / 10</label>
        <input type="range" id="lpSlider" min="0" max="10" step="1" value="0" oninput="drawSky()">

        <label>Moon (0 = new moon, 100 = full moon): <span id="moonValue">0</span></label>
        <input type="range" id="moonSlider" min="0" max="100" step="1" value="0" oninput="drawSky()">

        <label>Observatory altitude</label>
        <select id="altSelect" onchange="drawSky()">
            <option value="0">Sea level</option>
            <option value="1">Hillside</option>
            <option value="2">Mountain summit</option>
            <option value="3">High-altitude balloon</option>
        </select>

        <div class="info" id="skyInfo"></div>
    </div>

    <!-- Part D: telescope -->
    <div class="panel">
        <h3>Part D: Inspect the telescope</h3>
        <p>Click each part to find out what it does. Which part collects the light?</p>
        <svg viewBox="0 0 400 260" style="width:100%; max-width:400px; background:#e0eaff; border-radius:12px">
            <path class="part" id="tripod" d="M200 150 L140 250 M200 150 L260 250 M200 150 L200 250" stroke="#475569" stroke-width="8" fill="none" onclick="showPart('tripod', this)"/>
            <rect class="part" id="mount" x="185" y="125" width="30" height="30" fill="#334155" onclick="showPart('mount', this)"/>
            <g transform="rotate(-30 200 125)">
                <rect class="part" id="tube" x="100" y="110" width="190" height="30" fill="#1e40af" onclick="showPart('tube', this)"/>
                <rect class="part" id="objective" x="285" y="105" width="8" height="40" fill="#60a5fa" onclick="showPart('objective', this)"/>
                <rect class="part" id="eyepiece" x="78" y="118" width="22" height="14" fill="#111827" onclick="showPart('eyepiece', this)"/>
                <rect class="part" id="finder" x="150" y="90" width="70" height="12" fill="#D16200" onclick="showPart('finder', this)"/>
            </g>
        </svg>
        <div class="info" id="partInfo">Click a part of the telescope.</div>
        <p><small>Solar observing needs certified solar filters.</small></p>
    </div>

    <!-- Part E: key words -->
    <div class="panel">
        <h3>Key ideas</h3>
        <p><strong>Seeing:</strong> blurring of the image caused by moving air.</p>
        <p><strong>Extinction:</strong> light lost as it passes through the atmosphere.</p>
        <p><strong>Light pollution:</strong> artificial light that brightens the sky and hides faint objects.</p>
        <p><strong>Moonlight:</strong> a bright moon also brightens the sky and reduces contrast.</p>
        <p><strong>Why a balloon?</strong> Project SunbYte lifts its telescope above most of the atmosphere, so there is far less extinction and distortion. Try the balloon altitude in Part B.</p>
        <p><small>The numbers here are simplified teaching values, not real site data.</small></p>
    </div>

    <button onclick="location.href='#activities'">Back to activities</button>
</section>

<!-- ================= ACTIVITY: FABRY-PEROT ================= -->
<section id="fabry" class="activity">
    <h2>Fabry-Perot Interferometer</h2>

    <div class="panel">
        <h3>What is it?</h3>
        <p>Two parallel, partially reflecting plates, separated by a very small distance.</p>
        <svg viewBox="0 0 400 210" style="width:100%; max-width:500px">
            <rect x="130" y="20" width="8" height="160" fill="#93c5fd"/>
            <rect x="262" y="20" width="8" height="160" fill="#93c5fd"/>
            <line x1="30" y1="170" x2="134" y2="150" stroke="#D16200" stroke-width="3"/>
            <polyline points="134,150 266,130 134,110 266,90 134,70 266,50" fill="none" stroke="#D16200" stroke-width="2"/>
            <line x1="266" y1="130" x2="370" y2="110" stroke="#D16200" stroke-width="3"/>
            <line x1="266" y1="90" x2="370" y2="70" stroke="#D16200" stroke-width="3" opacity="0.6"/>
            <line x1="266" y1="50" x2="370" y2="30" stroke="#D16200" stroke-width="3" opacity="0.35"/>
            <line x1="134" y1="190" x2="266" y2="190" stroke="#172554" stroke-width="2"/>
            <text x="200" y="205" text-anchor="middle" font-size="14" fill="#172554">d</text>
            <text x="134" y="14" text-anchor="middle" font-size="12" fill="#172554">Plate 1</text>
            <text x="266" y="14" text-anchor="middle" font-size="12" fill="#172554">Plate 2</text>
        </svg>
        <div class="info">
            <p>Look at the zig-zag in the diagram. The light reflects back and forth between the plates, and a little escapes each time. All the escaping beams interfere. This gives very narrow transmission peaks.</p>
            <p><strong>Why is this useful?</strong> In solar observations we need sub-angstrom spectral resolution (1 angstrom = 0.1 nm) to examine very narrow absorption or emission features.</p>
        </div>
    </div>

    <div class="panel">
        <h3>The equation</h3>
        <p>A bright peak happens when the path difference is a whole number of wavelengths.</p>
        <div class="formula">2 n d cos(&theta;) = m &lambda;</div>
        <p><strong>n</strong> = refractive index between the plates</p>
        <p><strong>d</strong> = distance between the plates</p>
        <p><strong>&theta;</strong> = angle of the light inside the cavity</p>
        <p><strong>m</strong> = interference order (a whole number)</p>
        <p><strong>&lambda;</strong> = wavelength of the light</p>
    </div>

    <div class="panel">
        <h3>Change the wavelength it selects</h3>
        <div class="info">So the Fabry-Perot can select a very narrow range of wavelengths. How do we change which wavelength it selects? Use the sliders and watch the peaks move.</div>
        <div class="predict" style="margin-top:15px"><strong>Predict first:</strong> What happens to the peaks if you increase the plate spacing?</div>

        <p><strong>Transmission against wavelength</strong></p>
        <canvas id="peakCanvas" width="600" height="220"></canvas>

        <p><strong>The rings you would see</strong></p>
        <canvas id="fpCanvas" width="300" height="300"></canvas>
        <div class="info" id="fpInfo"></div>

        <label>Plate spacing: <span id="spacingValue"></span> micrometres</label>
        <input type="range" id="spacing" min="2" max="20" step="0.01" value="10" oninput="updateFabry()">

        <label>Refractive index: <span id="indexValue"></span></label>
        <input type="range" id="refractive" min="1" max="1.5" step="0.01" value="1" oninput="updateFabry()">

        <label>Incident angle: <span id="angleValue"></span> degrees</label>
        <input type="range" id="angle" min="0" max="20" step="1" value="0" oninput="updateFabry()">

        <label>Plate reflectivity: <span id="rValue"></span></label>
        <input type="range" id="reflectivity" min="0.1" max="0.98" step="0.01" value="0.88" oninput="updateFabry()">

        <div class="info">
            <p><strong>Bright:</strong> waves add up (constructive interference). <strong>Dark:</strong> waves cancel (destructive interference).</p>
            <p><strong>Reflectivity:</strong> higher reflectivity means more bounces, so the peaks and rings get sharper.</p>
        </div>
    </div>

    <button onclick="location.href='#activities'">Back to activities</button>
</section>

<footer>
    <p>&copy; 2026 SunbYte Observatory Labs (SOL), part of Project SunbYte, University of Sheffield</p>
    <p>Keep Learning. Keep Discovering.</p>
</footer>

<script>
    // =====================================================
    // ACTIVITY 1, PART A: WEATHER AND THE SUN
    // =====================================================

    var weatherText = {
        clear: "<strong>Clear:</strong> Light passes through the air with little scattering, so the Sun looks sharp.",
        cloudy: "<strong>Cloudy:</strong> Clouds block and scatter sunlight, so the Sun is hard to see.",
        rainy: "<strong>Rainy:</strong> Clouds and water droplets block the light. Observing is not possible.",
        hazy: "<strong>Hazy:</strong> Dust and tiny particles absorb and scatter light. This is extinction, so the image is dimmer and less contrasty.",
        turbulent: "<strong>Turbulent air:</strong> Pockets of warm and cool air bend light differently. This is poor seeing, so the image wobbles and blurs."
    };

    function setWeather(name, button) {
        var view = document.getElementById("sunView");
        view.className = "sun-view " + name;

        var buttons = document.getElementsByClassName("weatherBtn");
        for (var i = 0; i < buttons.length; i++) {
            buttons[i].classList.remove("active");
        }
        button.classList.add("active");

        document.getElementById("weatherInfo").innerHTML = weatherText[name];
    }

    function changeZoom() {
        var zoom = document.getElementById("zoom").value;
        document.getElementById("zoomValue").innerText = zoom;
        document.getElementById("sun").style.transform = "scale(" + zoom + ")";
    }

    // Show the starting text
    document.getElementById("weatherInfo").innerHTML = weatherText.clear;


    // =====================================================
    // ACTIVITY 1, PART B: NIGHT SKY
    // =====================================================

    // Make 500 random stars. A bigger "mag" means a fainter star.
    var stars = [];
    for (var i = 0; i < 500; i++) {
        stars.push({
            x: Math.random() * 600,
            y: Math.random() * 300,
            mag: 1 + 6 * Math.sqrt(Math.random())
        });
    }

    function drawSky() {
        var lp = Number(document.getElementById("lpSlider").value);
        var moon = Number(document.getElementById("moonSlider").value);
        var alt = Number(document.getElementById("altSelect").value);

        document.getElementById("lpValue").innerText = lp;
        document.getElementById("moonValue").innerText = moon;

        // Higher altitude = less extinction and steadier air
        var extinction = [0.4, 0.25, 0.1, 0.02][alt];
        var seeing = [3, 2, 0.8, 0.1][alt];

        // The faintest star magnitude you can see (bigger = better)
        var limit = 6.5 - 0.5 * lp - 2.5 * (moon / 100) - extinction;

        // Brighter sky background with more light pollution and moonlight
        var glow = Math.min(120, lp * 9 + moon * 0.4);

        var canvas = document.getElementById("skyCanvas");
        var ctx = canvas.getContext("2d");
        ctx.fillStyle = "rgb(" + (glow * 0.6) + "," + (glow * 0.7) + "," + (15 + glow) + ")";
        ctx.fillRect(0, 0, 600, 300);

        var count = 0;
        for (var i = 0; i < stars.length; i++) {
            var s = stars[i];
            if (s.mag < limit) {
                count++;
                ctx.beginPath();
                ctx.arc(s.x, s.y, seeing * 0.6 + (7 - s.mag) * 0.3, 0, 2 * Math.PI);
                ctx.fillStyle = "rgba(255,255,255," + (0.4 + (limit - s.mag) / 8) + ")";
                ctx.fill();
            }
        }

        document.getElementById("skyInfo").innerHTML =
            "<strong>Faintest star visible (limiting magnitude): " + limit.toFixed(1) + "</strong><br>" +
            "Stars you can see: " + count + "<br>" +
            "Blur from seeing: about " + seeing + " arcseconds";
    }

    drawSky();


    // =====================================================
    // ACTIVITY 2: FABRY-PEROT INTERFEROMETER
    // =====================================================

    function updateFabry() {
        // Read the sliders
        var d = Number(document.getElementById("spacing").value);
        var n = Number(document.getElementById("refractive").value);
        var angle = Number(document.getElementById("angle").value);
        var R = Number(document.getElementById("reflectivity").value);

        // Show the numbers next to the sliders
        document.getElementById("spacingValue").innerText = d.toFixed(2);
        document.getElementById("indexValue").innerText = n.toFixed(2);
        document.getElementById("angleValue").innerText = angle;
        document.getElementById("rValue").innerText = R.toFixed(2);

        drawFabry(d, n, angle, R);
        drawPeaks(d, n, angle, R);

        document.getElementById("fpInfo").innerHTML =
            "<strong>Your settings:</strong> spacing = " + d.toFixed(2) + " micrometres, n = " + n.toFixed(2) +
            ", angle = " + angle + " degrees, reflectivity = " + R.toFixed(2) + ".<br>" +
            "Bright rings = constructive interference. Dark rings = destructive interference.";
    }

    function drawFabry(d, n, angle, R) {
        var canvas = document.getElementById("fpCanvas");
        var ctx = canvas.getContext("2d");
        var size = canvas.width;
        var image = ctx.createImageData(size, size);
        var data = image.data;

        var centre = size / 2;
        var wavelength = 550e-9;                          // green light, in metres
        var finesse = (4 * R) / Math.pow(1 - R, 2);       // how sharp the rings are
        var maxAngle = 0.45;                              // field of view (radians)
        var tilt = angle * Math.PI / 180;                 // tilt of the light

        for (var y = 0; y < size; y++) {
            for (var x = 0; x < size; x++) {

                // Position of this pixel, from -1 to +1
                var dx = (x - centre) / centre;
                var dy = (y - centre) / centre;
                var radius = Math.sqrt(dx * dx + dy * dy);

                // Angle of the light outside the mirrors for this pixel
                var tx = dx * maxAngle + tilt;
                var ty = dy * maxAngle;
                var thetaOut = Math.sqrt(tx * tx + ty * ty);

                // Snell's law: angle of the light inside the cavity
                var thetaIn = Math.asin(Math.min(1, Math.sin(thetaOut) / n));

                // Phase difference between neighbouring beams
                var phase = (4 * Math.PI * n * d * 1e-6 * Math.cos(thetaIn)) / wavelength;

                // Airy function: how much light gets through
                var transmission = 1 / (1 + finesse * Math.pow(Math.sin(phase / 2), 2));

                // Fade to black at the edge of the circle
                var fade = Math.max(0, 1 - Math.pow(radius, 8));
                var brightness = Math.round(255 * transmission * fade);

                var i = (y * size + x) * 4;
                data[i] = Math.round(brightness * 0.65);      // red
                data[i + 1] = Math.round(brightness * 0.75);  // green
                data[i + 2] = brightness;                     // blue
                data[i + 3] = 255;                            // opacity
            }
        }

        ctx.putImageData(image, 0, 0);
    }


    // =====================================================
    // SPECTROMETER: INTEGRATION TIME
    // =====================================================

    var wavelengths = [];
    for (var w = 400; w <= 700; w++) { wavelengths.push(w); }

    // Fixed random noise, so the graph does not flicker
    var noise = wavelengths.map(function () { return (Math.random() + Math.random() + Math.random() - 1.5) * 2; });

    var maxCount = 4000;   // detector limit (photons per pixel)

    function bump(w, centre, width) { return Math.exp(-0.5 * Math.pow((w - centre) / width, 2)); }

    // Brightness (0 to 1) of each light source at wavelength w
    function rate(src, w) {
        var v = 0;
        if (src == "sun") {
            v = 0.3 + 0.7 * bump(w, 520, 120);
            var dips = [[393, 0.7], [430, 0.35], [486, 0.5], [527, 0.25], [589, 0.6], [656, 0.6]];
            var dip = 0;
            for (var i = 0; i < dips.length; i++) { dip += dips[i][1] * bump(w, dips[i][0], 1.5); }
            v = v * Math.max(0, 1 - dip);
        } else if (src == "hydrogen") {
            v = 0.03 + 0.3 * bump(w, 410, 1.2) + 0.4 * bump(w, 434, 1.2) + 0.6 * bump(w, 486, 1.2) + bump(w, 656, 1.2);
        } else if (src == "sodium") {
            v = 0.03 + bump(w, 589, 1.2) + 0.15 * bump(w, 568, 1) + 0.1 * bump(w, 616, 1);
        } else {
            v = 0.02 + 0.8 * bump(w, 450, 10) + 0.6 * bump(w, 560, 50);
        }
        return v;
    }

    var sourceText = {
        sun: "<strong>The Sun:</strong> a smooth continuum with dark absorption lines, where gas in the Sun's atmosphere absorbs light.",
        hydrogen: "<strong>Hydrogen lamp:</strong> only a few bright emission lines. The red one is at 656 nm.",
        sodium: "<strong>Sodium lamp:</strong> one very bright yellow emission line near 589 nm.",
        led: "<strong>White LED:</strong> a blue peak plus a broad hump from the yellow phosphor. No sharp lines."
    };

    function drawSpectrum() {
        var t = Number(document.getElementById("tSlider").value);
        var src = document.getElementById("srcSelect").value;
        document.getElementById("tValue").innerText = t;

        var ctx = document.getElementById("specCanvas").getContext("2d");
        var left = 55, right = 590, top = 15, bottom = 265;
        ctx.fillStyle = "#0b1020";
        ctx.fillRect(0, 0, 600, 300);

        // Axes
        ctx.strokeStyle = "white"; ctx.fillStyle = "white"; ctx.font = "12px Arial"; ctx.lineWidth = 1;
        ctx.beginPath(); ctx.moveTo(left, top); ctx.lineTo(left, bottom); ctx.lineTo(right, bottom); ctx.stroke();
        ctx.textAlign = "center";
        for (var w = 400; w <= 700; w += 50) {
            var tx = left + (w - 400) / 300 * (right - left);
            ctx.fillText(w, tx, bottom + 15);
        }
        ctx.fillText("Wavelength (nm)", (left + right) / 2, 292);
        ctx.textAlign = "right";
        for (var c = 0; c <= maxCount; c += 1000) {
            var ty = bottom - c / maxCount * (bottom - top);
            ctx.fillText(c, left - 5, ty + 4);
        }
        ctx.save(); ctx.translate(12, 140); ctx.rotate(-Math.PI / 2); ctx.textAlign = "center";
        ctx.fillText("Photon count", 0, 0); ctx.restore();

        // Detector limit line
        ctx.strokeStyle = "#f87171"; ctx.setLineDash([5, 5]);
        ctx.beginPath(); ctx.moveTo(left, top); ctx.lineTo(right, top); ctx.stroke();
        ctx.setLineDash([]);

        // Spectrum
        var peak = 0, clipped = 0;
        ctx.strokeStyle = "#60a5fa"; ctx.lineWidth = 1.5; ctx.beginPath();
        for (var i = 0; i < wavelengths.length; i++) {
            var r = rate(src, wavelengths[i]);
            var counts = r * t * 50;                                   // photons collected
            if (counts > peak) { peak = counts; }
            var measured = counts + noise[i] * (Math.sqrt(counts) + 5); // add noise
            measured = Math.max(0, measured);
            var isClipped = measured >= maxCount;
            if (isClipped) { measured = maxCount; clipped++; }
            var x = left + (wavelengths[i] - 400) / 300 * (right - left);
            var y = bottom - measured / maxCount * (bottom - top);
            if (i == 0) { ctx.moveTo(x, y); } else { ctx.lineTo(x, y); }
        }
        ctx.stroke();

        var status;
        if (peak >= maxCount) { status = "<strong style='color:#b91c1c'>Too long:</strong> the brightest pixels are clipped at the detector limit, so the true peak height is lost."; }
        else if (peak < 1500) { status = "<strong style='color:#b45309'>Too short:</strong> too few photons, so the signal is noisy."; }
        else { status = "<strong style='color:#15803d'>Good:</strong> smooth signal and no clipping."; }

        document.getElementById("specInfo").innerHTML = sourceText[src] + "<br><br>" + status;
    }

    drawSpectrum();


    // =====================================================
    // OBSERVATORY: BALLOON AND TELESCOPE
    // =====================================================

    var levels = {
        ground: { bottom: "4%", blur: 6, opacity: 0.5, text: "<strong>Ground:</strong> light crosses the whole atmosphere. There is a lot of extinction and seeing blur, so the image is dim and soft." },
        low: { bottom: "38%", blur: 3, opacity: 0.75, text: "<strong>A little above ground:</strong> there is less air above the telescope, so the image is a bit brighter and sharper." },
        high: { bottom: "80%", blur: 0.5, opacity: 1, text: "<strong>Balloon height (about 30 km):</strong> the telescope is above most of the atmosphere. There is very little extinction or blur. This is what Project SunbYte aims for." }
    };

    function setLevel(name, button) {
        var l = levels[name];
        document.getElementById("balloon").style.bottom = l.bottom;
        var sun = document.getElementById("sun2");
        sun.style.filter = "blur(" + l.blur + "px)";
        sun.style.opacity = l.opacity;
        var buttons = document.getElementsByClassName("lvlBtn");
        for (var i = 0; i < buttons.length; i++) { buttons[i].classList.remove("active"); }
        button.classList.add("active");
        document.getElementById("levelInfo").innerHTML = l.text;
    }

    setLevel("ground", document.getElementById("lvlGround"));

    var partText = {
        objective: "<strong>Objective (lens or mirror):</strong> collects the light and brings it to a focus.",
        tube: "<strong>Tube:</strong> holds the optics in line and keeps stray light out.",
        finder: "<strong>Finder scope:</strong> a small wide-view telescope used to point at the target.",
        eyepiece: "<strong>Eyepiece (or camera):</strong> magnifies the image, or holds a detector.",
        mount: "<strong>Mount:</strong> lets the telescope turn and track the Sun as it moves.",
        tripod: "<strong>Tripod:</strong> holds everything steady. Vibrations blur the image."
    };

    function showPart(name, element) {
        var parts = document.getElementsByClassName("part");
        for (var i = 0; i < parts.length; i++) { parts[i].classList.remove("chosen"); }
        element.classList.add("chosen");
        document.getElementById("partInfo").innerHTML = partText[name];
    }


    // =====================================================
    // FABRY-PEROT: TRANSMISSION PEAKS
    // =====================================================

    function drawPeaks(d, n, angle, R) {
        var ctx = document.getElementById("peakCanvas").getContext("2d");
        var left = 55, right = 590, top = 15, bottom = 180;
        ctx.fillStyle = "#0b1020";
        ctx.fillRect(0, 0, 600, 220);

        ctx.strokeStyle = "white"; ctx.fillStyle = "white"; ctx.font = "12px Arial"; ctx.lineWidth = 1;
        ctx.beginPath(); ctx.moveTo(left, top); ctx.lineTo(left, bottom); ctx.lineTo(right, bottom); ctx.stroke();
        ctx.textAlign = "center";
        for (var w = 520; w <= 580; w += 10) {
            ctx.fillText(w, left + (w - 520) / 60 * (right - left), bottom + 15);
        }
        ctx.fillText("Wavelength (nm)", (left + right) / 2, 212);
        ctx.textAlign = "right";
        ctx.fillText("1", left - 5, top + 4);
        ctx.fillText("0", left - 5, bottom + 4);
        ctx.save(); ctx.translate(12, 100); ctx.rotate(-Math.PI / 2); ctx.textAlign = "center";
        ctx.fillText("Transmission", 0, 0); ctx.restore();

        var finesse = (4 * R) / Math.pow(1 - R, 2);
        var thetaIn = Math.asin(Math.min(1, Math.sin(angle * Math.PI / 180) / n));

        ctx.strokeStyle = "#60a5fa"; ctx.lineWidth = 1.5; ctx.beginPath();
        for (var k = 0; k <= 1200; k++) {
            var lambda = 520 + k * 0.05;   // nm
            var half = 2 * Math.PI * n * d * 1e-6 * Math.cos(thetaIn) / (lambda * 1e-9);
            var T = 1 / (1 + finesse * Math.pow(Math.sin(half), 2));
            var x = left + k / 1200 * (right - left);
            var y = bottom - T * (bottom - top);
            if (k == 0) { ctx.moveTo(x, y); } else { ctx.lineTo(x, y); }
        }
        ctx.stroke();
    }

    updateFabry();
</script>

</body>
</html>
"""

# ------------------------------------------------------------
# SAVE THE WEBSITE AS index.html
# ------------------------------------------------------------

Path("index.html").write_text(website, encoding="utf-8")

# ------------------------------------------------------------
# START A SMALL WEB SERVER AND SHOW IT IN THE NOTEBOOK
# ------------------------------------------------------------

class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # stops the server printing lots of messages

server = None
for port in range(8000, 8100):
    try:
        server = HTTPServer(("0.0.0.0", port), QuietHandler)
        break
    except OSError:
        continue  # port already used, try the next one

if server is None:
    raise RuntimeError("No available port found!")

threading.Thread(target=server.serve_forever, daemon=True).start()

output.serve_kernel_port_as_iframe(server.server_port, height=1000)
print("SOL Science website is running!")

<IPython.core.display.Javascript object>

SOL Science website is running!
